# Define a tool in code, or in the catalog

Every tool you give a model has two halves.

The **definition** is what the model reads to decide whether and how to call the tool: its
name, its description, and the JSON Schema of its parameters. The **implementation** is the
code that runs when the model asks for it.

Both halves have to live somewhere, and they do not have to live in the same place. That is
the whole subject of this notebook. AcruxCore supports two answers:

| | Who owns the definition | How you run it |
|---|---|---|
| **Path A** | your code — `@acrux.tool` derives it from the function | `tools=[fn]` |
| **Path B** | the catalog — a version you commit in the dashboard | `client_tools={"name": fn}` |

You will build the **same weather tool twice**, once each way, and then look at what
actually changed: who decides the parameter names, whether a version pin survives, and what
your traces can tell you afterwards.

Everything below runs against a real account. Nothing is mocked.

**Three kinds of code cell.** Most of these cells are setup or checks, not code you would
ship. Every cell's lead-in says which kind it is, so you know what to copy into a project of
your own:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |

The two paths ship different code, and Step 6 puts both side by side at the end.

**To run it:** clone the repo, `pip install acruxcore`, and open this file in Jupyter.

**Companion guide:** [Define a tool in code or in the catalog](https://docs.acruxcore.com/docs/guides/define-a-tool-in-code-or-in-the-catalog).

---

## Step 0 — What you need before running anything

If this is a brand new account, do these four things first. Each one takes a minute.

**1. Create an account** at [acruxcore.com](https://acruxcore.com) and verify your email.

**2. Create a personal API key.** In the dashboard, go to **Account & keys → New key**, give
it a name, and copy the value. It is shown once and stored hashed, so if you lose it, make
another.

**3. Connect a model.** A fresh account has no way to reach an LLM yet, and this is the step
people skip. Two parts, both in the dashboard:

- **Gateway → Credentials → New credential** — paste a provider key (an OpenAI or
  OpenRouter key, for example).
- **Gateway → Models → New model** — give it a **public name** and point it at the
  credential. The public name is what you pass as the model in code. This notebook uses
  `gpt-4o-mini`; if yours is called something else, change `MODEL` in the next cell.

See [Route your app's LLM calls through the gateway](https://docs.acruxcore.com/docs/guides/route-calls-through-the-gateway)
if you want that step with screenshots.

**4. Install the SDK.**

In [ ]:
# Step 3 of this notebook needs client_tools, which landed after 0.9.0.
%pip install -q --upgrade acruxcore

**Setup.** Now set your key and the base URL. Prefer real environment variables — the two `os.environ`
lines below are here so the notebook is self-contained, but a key pasted into a notebook
also gets saved into the notebook.

In [1]:
import json
import os

# Set these in your shell instead if you can: ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

MODEL = "gpt-4o-mini"          # a public name from Gateway -> Models
PROMPT_A = "weather-brief-code"        # path A: no tool bound to it
PROMPT_B = "weather-brief-catalog"     # path B: one tool bound, pinned
TOOL_B = "get_weather_catalog"


### Preflight

**Check.** Run this before anything else. It checks the four things above in the order they fail, so
you get one clear message instead of a stack trace from deep inside a tool loop.

In [2]:
import inspect

import httpx

import acruxcore
from acruxcore import AcruxCore
from acruxcore.gateway_api import GatewayNamespace

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the key work at all?
await hub.prompts.list(limit=1)
print("api key: ok")

# 2. Does this SDK support client_tools? Step 3 needs it; Step 2 does not.
has_client_tools = "client_tools" in inspect.signature(
    GatewayNamespace.run_prompt_with_tools
).parameters
print("client_tools supported:", has_client_tools)
if not has_client_tools:
    print(f"  !! acruxcore {acruxcore.__version__} is too old for Step 3 - upgrade it")

# 3. Is there a model to run on, and is MODEL one of them?
#    There is no SDK method for this yet, so call the endpoint directly.
async with httpx.AsyncClient() as http:
    res = await http.get(
        f"{os.environ['ACRUXCORE_BASE_URL']}/gateway/models",
        headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    )
models = [m["publicName"] for m in res.json()]
print("models on this team:", models or "NONE - add one in Gateway -> Models")
print(f"MODEL {MODEL!r} available:", MODEL in models)

api key: ok
client_tools supported: True
models on this team: ['gpt-4o-mini']
MODEL 'gpt-4o-mini' available: True


---

## Step 1 — What "the definition" actually is

Before the code, be precise about what changes hands. The definition is exactly the object
sent to the model on every call, plus two facts that travel beside it:

| Part | Who reads it | Why it matters |
|---|---|---|
| `name` | the model | how the model refers to the tool |
| `description` | the model | whether the model picks this tool at all |
| `parameters` (JSON Schema) | the model | which arguments it may send, and which are required |
| `executor` | the platform | whether your process runs the tool, or the gateway calls a URL |
| version identity | the platform | which build ran, and what lands on the trace |

The parameter schema is the one with teeth. The name and description only influence
*whether* the tool gets picked; the schema decides the **shape of the call**. So "who owns
the definition" really means "who decides the call shape, and who has to fit it".

One consequence up front: a decorator wraps a Python function, so it can only ever produce
a `client` executor — your process runs the body. Only a catalog-defined tool can be
`http`, where the gateway calls a URL and your code does nothing.

---

## Step 2 — Path A: your code owns the definition

Everything in this step lives in your codebase. The catalog only receives a copy.

### 2a. Declare the tool

`@acrux.tool` reads the function and attaches the definition to it. The name comes from the
function name, the description from the first line of the docstring, and the parameter
schema from the type hints.

The decorator is **pure**: no network call happens at import time. That is deliberate —
a decorator that registered on import would make importing a module hit the network and
would fire during `--help`.

**Your app.** On path A the decorated function is the definition *and* the implementation,
so the next cell is the one thing here that really ships. There is no dashboard route for
this step — writing the function *is* the route.

In [3]:
from acruxcore import acrux


@acrux.tool
async def get_weather_code(city: str) -> dict:
    """Get today's weather for a city.

    Args:
        city: City name, e.g. 'Lahore'.
    """
    print(f"  -> get_weather_code(city={city!r})")
    return {"city": city, "temp_c": 34, "sky": "hazy sun"}

### 2b. What the decorator produced

**Check.** The function is returned unchanged apart from one added attribute, so it stays
directly callable and testable. That attribute holds the whole definition:

In [4]:
spec = get_weather_code.__acrux_tool__

print("name:       ", spec.name)
print("description:", repr(spec.description), " <- the docstring's first line")
print("executor:   ", spec.executor, " <- a decorator can only produce this")
print("parameters: ")
print(json.dumps(spec.parameters_schema, indent=2))

name:        get_weather_code
description: "Get today's weather for a city."  <- the docstring's first line
executor:    {'type': 'client'}  <- a decorator can only produce this
parameters: 
{
  "type": "object",
  "properties": {
    "city": {
      "type": "string",
      "description": "City name, e.g. 'Lahore'."
    }
  },
  "required": [
    "city"
  ]
}


Notice what you did **not** write: no JSON Schema, no `required` list, no parameter
description. `city: str` and the `Args:` block produced all of it.

A function with **no docstring** sends no description at all, which hands ownership of the
model-facing text back to the dashboard. Write a docstring when you want the code to own
that text.

### 2c. The prompt for path A has no tools

**Setup.** This is the part that surprises people. On path A the prompt knows nothing about
tools — the tool reaches the model straight from your process. Create it if it does not
exist:

In [5]:
found = [p for p in (await hub.prompts.list(search=PROMPT_A)).data if p.name == PROMPT_A]
if found:
    prompt_a_id = found[0].id
    print("prompt already exists:", PROMPT_A)
else:
    created = await hub.prompts.create(
        name=PROMPT_A, description="Weather brief, tool defined in code."
    )
    prompt_a_id = created.id
    await hub.prompts.commit_version(
        prompt_a_id,
        messages=[{
            "role": "system",
            "content": "You are a weather assistant. Call the tool, then answer in one sentence.",
        }],
        model=MODEL,
    )
    print("created prompt:", PROMPT_A)

rendered_a = await hub.prompts.render(PROMPT_A, "production")
print("model bound to the version:", rendered_a.model)
print("tools bound to this prompt:", rendered_a.tools, " <- empty, on purpose")

prompt already exists: weather-brief-code
model bound to the version: gpt-4o-mini
tools bound to this prompt: []  <- empty, on purpose


### 2d. Publish the definition

**Setup**, and in a real project a *deploy-step* one — never something on the request path.
Registration is a separate, explicit act. `tools.sync` reconciles the catalog with your
code: it creates the tool if needed, commits a version from the derived schema, and moves
the alias. It is idempotent and cached per process on the definition's hash, so calling it
twice with an unchanged function makes one request.

In [6]:
result = (await hub.tools.sync([get_weather_code]))[0]
print(f"tool_id={result.tool_id}")
print(f"version={result.version_number}  alias={result.alias}  committed_by_this_call={result.committed}")

tool_id=b93f5e0f-73fe-44c1-a988-8979839d6ab7
version=1  alias=production  committed_by_this_call=False


Open **Gateway → Tools → get_weather_code** in the dashboard and the catalog tells you where
that definition came from:

![Tool detail page for get_weather_code with a "Defined in code" badge under the name and one version v1 tagged "code"](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/01-tool-defined-in-code.png)

The **Defined in code** badge and the `code` tag on the version are not decoration. They
record that this version was written by `tools.sync`, which means editing the function is
how you change it. A version committed in the dashboard is tagged `dashboard`, and one
committed over the API is tagged `api`.

### 2e. Run it

**Your app.** `tools=[fn]` passes the definition inline, from your process. `sync=False` here only because
the cell above already published it — the default is `True`, which publishes on first use.

In [7]:
run_a = await hub.gateway.run_tool_loop(
    rendered_a.model,
    [*rendered_a.messages, {"role": "user", "content": "What is the weather in Karachi?"}],
    tools=[get_weather_code],
    sync=False,
    prompt_version_id=rendered_a.version_id,   # keeps prompt lineage on the trace
    trace={"name": "notebook-code-defined-tool"},
)
print("answer:", run_a.content)
print("trace: ", run_a.trace_id)

  -> get_weather_code(city='Karachi')
answer: The weather in Karachi is currently 34°C with hazy sun.
trace:  9087a6f9-3fb4-4efd-98d8-41eaa3a3e380


---

## Step 3 — Path B: the catalog owns the definition

Now the same tool, the other way round. Nothing in your code will define it, so the
definition has to exist in the platform **before** the run.

A tool is created in **two** steps, and this trips people up: a **shell** carries the name,
and a **version** carries the schema and the executor. A shell on its own is not callable —
its page says "No versions yet".

The rest of this step is one path, not two. First the dashboard route — three screens that
define the tool — then the same three things in code, for anyone who would rather not click.
What follows those is your app: read the definition, supply the body, run.

### In the dashboard

#### The shell

**Gateway → Tools → New tool.** Just a name and an optional description.

| Field | What to enter |
|---|---|
| **Name** | `get_weather_catalog` |
| **Description** | Weather lookup. |

![New tool dialog with the name field set to get_weather_catalog and the description "Weather lookup."](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/02-new-tool-shell.png)

#### The version

**New version** on the tool's page. The description here is the text the model reads. Each
parameter is one row — name, type, description, and whether it is required. The executor
stays **Client**, which means your app runs the body.

| Field | What to enter |
|---|---|
| **Description** | Get today's weather for a city. |
| **Changelog** | leave it empty — a note for your team, never sent to the model |
| **Parameter** | name `city`, type `string`, description *City name, e.g. 'Lahore'.*, required ✓ |
| **Executor** | Client — the caller's app runs it |

![New version dialog with description "Get today's weather for a city.", one parameter row named city of type string marked required, and the Executor select showing "Client - the caller's app runs it"](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/03-commit-tool-version.png)

Committed, the tool looks like this. Compare it with the screenshot in Step 2: same tool,
same schema, but no "Defined in code" badge, and the version is tagged `dashboard`.

![Tool detail page for get_weather_catalog with one version v1 tagged "dashboard" and no "Defined in code" badge](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/04-tool-defined-in-catalog.png)

#### Bind it to the prompt

A definition that nothing points at is never called. On the prompt's **Tools** tab, connect
the tool. A cell can follow a tool alias or pin an exact version — this one pins **v1**, so
the prompt keeps running that build even after someone commits v2.

The prompt itself is one system message, committed with a default model:

| Field | What to enter |
|---|---|
| **Name** | `weather-brief-catalog` |
| **Description** | Weather brief, tool defined in the catalog. |
| **Message** | one message, role **system**: *You are a weather assistant. Call the tool, then answer in one sentence.* |
| **Default model** | `gpt-4o-mini`, or whatever your **Gateway → Models** page calls it |
| **Tools tab** | connect `get_weather_catalog`, and set the default column to **pinned v1** |

![Prompt Tools tab for weather-brief-catalog showing one row, get_weather_catalog, with the default column set to "pinned v1"](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/05-bind-tool-pinned.png)

### The same thing in code

**Setup.** If you would rather not click, the cell below does exactly what those three
screens did — same shell, same version, same pinned binding. In a real project it runs
**once**, and it does not ship with your agent, which is the point of catalog-first. It is
written find-or-create, so running it twice is a no-op.

Every value in the tables above appears in it too, so the two routes cannot drift apart.

In [8]:
tools = [t for t in (await hub.tools.list(search=TOOL_B)).data if t.name == TOOL_B]
if tools:
    tool_b_id = tools[0].id
    print("tool already in the catalog:", TOOL_B)
else:
    shell = await hub.tools.create(name=TOOL_B, description="Weather lookup.")
    tool_b_id = shell.id
    version = await hub.tools.commit_version(
        tool_b_id,
        parameters_schema={
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Lahore'."}
            },
            "required": ["city"],
        },
        executor={"type": "client"},          # "the caller's own app runs it"
        description="Get today's weather for a city.",
    )
    print(f"created tool {TOOL_B} v{version.version_number}")

found = [p for p in (await hub.prompts.list(search=PROMPT_B)).data if p.name == PROMPT_B]
if found:
    prompt_b_id = found[0].id
    print("prompt already exists:", PROMPT_B)
else:
    created = await hub.prompts.create(
        name=PROMPT_B, description="Weather brief, tool defined in the catalog."
    )
    prompt_b_id = created.id
    print("created prompt:", PROMPT_B)

# Checked separately from creating: a prompt shell with zero versions is a real
# state, and "the name exists" is not "it has content".
if (await hub.prompts.list_versions(prompt_b_id)).total == 0:
    await hub.prompts.commit_version(
        prompt_b_id,
        messages=[{
            "role": "system",
            "content": "You are a weather assistant. Call the tool, then answer in one sentence.",
        }],
        model=MODEL,
    )
    print("committed prompt v1")

await hub.prompts.set_tool_binding(prompt_b_id, tool_b_id, pinned_version_number=1)
bindings = await hub.prompts.list_tool_bindings(prompt_b_id)
print("bound tools:", [(b.tool_name, f"pinned v{b.pinned_version_number}") for b in bindings.default])

tool already in the catalog: get_weather_catalog
prompt already exists: weather-brief-catalog
bound tools: [('get_weather_catalog', 'pinned v1')]


### Read the definition back

**Check.** This is the same tool, but now the definition is a platform fact. Your code can
read it — and cannot change it by running:

In [9]:
resolved = (await hub.tools.resolve([{"name": TOOL_B, "version": 1}]))[0]

print("executor:", resolved.executor_type, " version:", resolved.version_number)
print(json.dumps(resolved.function, indent=2))

executor: client  version: 1
{
  "name": "get_weather_catalog",
  "description": "Get today's weather for a city.",
  "parameters": {
    "type": "object",
    "required": [
      "city"
    ],
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. 'Lahore'."
      }
    }
  }
}


### The implementation, and only the implementation

**Your app.** No decorator. No schema. Nothing published. Just a function, and a map saying
which catalog tool it implements:

In [10]:
def get_weather(city: str) -> dict:
    """For humans reading this notebook. The model never sees this docstring --
    the description it reads is the one stored on the catalog version."""
    print(f"  -> get_weather(city={city!r})")
    return {"city": city, "temp_c": 34, "sky": "hazy sun"}


# The key is the catalog tool's name -- the string the dashboard shows, and the
# name the model calls. The value is any function you like: this one is called
# get_weather, while the tool it implements is called get_weather_catalog.
CLIENT_TOOLS = {TOOL_B: get_weather}

for tool_name, fn in CLIENT_TOOLS.items():
    print(f"catalog tool {tool_name!r} -> python function {fn.__name__!r}")

catalog tool 'get_weather_catalog' -> python function 'get_weather'


#### How your function gets matched to the tool

The **key** is the whole wiring. Nothing else takes part — not the function's name, not the
module it lives in, not the order of the entries. At run time the name travels like this:

```
the prompt's binding  ->  the catalog tool's name  ->  your map's key  ->  your function
```

The model asks for the tool by that same catalog name, so the key has to match what the
dashboard shows, exactly, including case. That is why the cell above prints the pair: the
left side is a platform fact, the right side is just a Python object.

The **value** is any callable you like. Here the tool is `get_weather_catalog` while the
function is `get_weather`, and that is deliberate — the platform owns one name, your
codebase owns the other, and the map is the one place they meet. Renaming the Python
function changes nothing on the platform. Renaming the tool in the dashboard means updating
one string here.

With several tools it is one entry each:

```python
CLIENT_TOOLS = {
    "get_weather_catalog": lookup_weather,
    "search_flights": find_flights,
    "convert_currency": fx,
}
```

Write the keys as literal strings rather than deriving them from `fn.__name__`. The file
then states which catalog tools this app implements, and it keeps working when an
implementation is a wrapped function or a `functools.partial` — neither of which has a name
you can rely on.

Two more rules follow from the same idea, that the definition belongs to the catalog:

- **The parameter names are not yours either.** The function is called with the schema's own
  field names as keywords, so `lookup_weather(city=...)`. A function that cannot accept
  `city` is rejected before the first model call, not halfway through the loop.
- **Only `client` tools belong in the map.** A prompt's `http` tools run on the platform and
  need nothing from you. A key that matches nothing bound to the prompt is ignored, so one
  app-wide map can serve several prompts.

### Run it

`run_prompt_with_tools` takes everything else from the render: the model from the version's
bound model, the tools from the prompt's bindings, and the prompt version id that keeps
trace lineage intact. That last one is the easy one to forget by hand, and forgetting it
costs lineage silently, because the call still works.

**Your app.** `client_tools` needs an `acruxcore` newer than 0.9.0. If the preflight printed
`client_tools supported: False`, this is the cell that will fail — upgrade and re-run.

In [11]:
rendered_b = await hub.prompts.render(PROMPT_B, "production")
print("tools bound to this prompt:", [t["function"]["name"] for t in rendered_b.tools])

run_b = await hub.gateway.run_prompt_with_tools(
    rendered_b,
    messages=[*rendered_b.messages, {"role": "user", "content": "What is the weather in Karachi?"}],
    client_tools=CLIENT_TOOLS,
    trace={"name": "notebook-catalog-defined-tool"},
)
print("answer:", run_b.content)
print("trace: ", run_b.trace_id)

tools bound to this prompt: ['get_weather_catalog']
  -> get_weather(city='Karachi')
answer: The weather in Karachi is 34°C with a hazy sun.
trace:  e0155da9-3fe0-4381-a23d-67de64a63580


---

## Step 4 — What actually changed

Two runs, same answer, same tool. Here is everything the choice moved:

| | Code owns it (`@acrux.tool`) | Catalog owns it (`client_tools`) |
|---|---|---|
| Schema comes from | your type hints | the catalog version |
| Description comes from | your docstring | the catalog version |
| Parameter names | your function decides | the schema decides, your function must fit |
| Executor | always `client` | `client` or `http` |
| Changing what the model reads | edit the function, sync, redeploy | edit a version in the dashboard, no deploy |
| Version pin on a prompt | dropped when `tools=[fn]` syncs | kept, and travels as a pin |
| Trace tool span | stamped only when the loop syncs | always stamped with `toolId:version` |

That last row is checkable. The tool span from the catalog run carries the exact version
that ran:

![Trace detail for catalog-defined-tool, with the tool span expanded showing three attributes - the city argument, executorType client, and a toolVersionId ending in colon one](../../../../apps/docs/static/img/tutorials/define-a-tool-in-code-or-in-the-catalog/06-trace-tool-version-stamp.png)

**Check.** Rather than trust the screenshot, read it back from the API. Traces are reported
in the background, so give it a moment to arrive.

`walk` and `tool_spans` in the next cell are helpers this notebook defines, not SDK
functions. The real call inside them is `hub.traces.get()`; the rest is a loop over the span
tree and a short poll, because a trace takes a moment to land.

In [12]:
import asyncio


def walk(spans):
    for s in spans:
        yield s
        yield from walk(s.children)


async def tool_spans(trace_id, attempts=15):
    """Traces are reported off the critical path, so poll briefly."""
    for _ in range(attempts):
        detail = await hub.traces.get(trace_id)
        found = [s for s in walk(detail.spans) if s.kind == "tool"]
        if found:
            return found
        await asyncio.sleep(1)
    return []


for label, trace_id in [("path A (code)", run_a.trace_id), ("path B (catalog)", run_b.trace_id)]:
    for span in await tool_spans(trace_id):
        attrs = span.attributes or {}
        print(f"{label:18} {span.name:22} toolVersionId={attrs.get('toolVersionId')!r}")

path A (code)      get_weather_code       toolVersionId=None
path B (catalog)   get_weather_catalog    toolVersionId='d951faf4-a7fc-415c-be57-4630582020c9:1'


Path B names an exact catalog version. Path A shows `None`, because this run used
`sync=False` — there is no catalog version that this particular run can honestly point at.
Run it with the default `sync=True` and the stamp appears.

---

## Step 5 — Four ways to get this wrong

Two of these you can trigger right here. The other two are described but deliberately not
run, because one of them would rewrite your catalog.

### Trap 1 — a catalog tool that nothing points at fails **silently**

If the definition exists but no binding and no `tool_refs` name it, `render` returns no
tools, and the run becomes a plain completion. The model answers from its own knowledge, no
error is raised, and your function is never called. Path A's prompt shows the shape of it —
this cell and the next one are **broken on purpose**, so neither is app code:

In [13]:
no_tools = await hub.prompts.render(PROMPT_A, "production")
print("tools:", no_tools.tools)

silent = await hub.gateway.run_prompt_with_tools(
    no_tools,
    messages=[*no_tools.messages, {"role": "user", "content": "What is the weather in Karachi?"}],
    client_tools=CLIENT_TOOLS,      # supplied, and never used
    trace={"name": "notebook-trap-no-binding"},
)
print("answer:", silent.content)
print("^ no error and no tool call - the model had nothing to call")

tools: []
answer: I currently don't have access to real-time weather data. Please check a reliable weather website or app for the latest information on Karachi's weather.
^ no error and no tool call - the model had nothing to call


If a tool "does nothing", check the prompt's **Tools** tab before you debug your code.

### Trap 2 — a bound `client` tool with no implementation fails **loudly**

**Broken on purpose.** This one is easy to fix, because the error names the keys you did
supply. It is raised before the first model call, so you are not billed for a round trip:

In [14]:
from acruxcore.errors import AcruxCoreError

try:
    await hub.gateway.run_prompt_with_tools(
        rendered_b,
        messages=[*rendered_b.messages, {"role": "user", "content": "Weather in Karachi?"}],
        client_tools={"get_wether_catalog": get_weather},   # deliberate typo
        trace=False,
    )
except AcruxCoreError as exc:
    print("code:", exc.code)
    print(exc)

code: MISSING_DISPATCH
acruxcore: tool 'get_weather_catalog' has a client executor, so something has to run it, but no implementation was supplied. Pass it in client_tools={'get_weather_catalog': ...}, or pass dispatch=. client_tools held: ['get_wether_catalog'].


### Trap 3 — `tools=[fn]` on a name that already exists **rewrites it**

Passing a decorated function whose name matches a catalog tool commits a new version from
your local schema and moves its alias — and a prompt that pinned an exact version **loses
the pin**. The run succeeds, so nothing looks wrong; the pinned prompt has quietly started
following whatever is on your laptop.

This notebook does not run that, on purpose. The rule to remember:

> `client_tools` runs a tool and writes nothing. `tools=[fn]` publishes a definition. Reach
> for `tools=` only when your code is meant to be the source of truth for that tool.

### Trap 4 — a decorated function inside `client_tools` keeps its decorator, and loses it

It runs fine, but the definition is ignored: the schema and description come from the
catalog. Someone can edit the docstring, redeploy, and wonder why the model's behaviour
never changed.

---

## Step 6 — Which one should you use?

**Default to the catalog owning the definition** whenever a prompt binds its tools. Version
pinning then means something, the model-facing text can be fixed without a deploy, and the
same prompt can run a `client` tool in staging and an `http` tool in production without you
touching the app.

**Choose the decorator when the tool exists only in code** and the repository should be the
source of truth — an internal agent, a CLI, no dashboard in the loop. Deriving a schema from
type hints is worth a lot when the tool is yours alone.

**The two combine well.** Run `tools.sync([...])` in your deploy step so definitions are
published from code, and run with `client_tools` at runtime so execution stays pinned to a
catalog version. You get schemas generated from code *and* traces that name the exact
version that ran.

### What of this actually ships

Strip out the setup and the checks and each path is short. Side by side:

```python
# Path A — your code owns the definition
@acrux.tool
async def get_weather_code(city: str) -> dict:
    """Get today's weather for a city.

    Args:
        city: City name, e.g. 'Lahore'.
    """
    return {"city": city, "temp_c": 34, "sky": "hazy sun"}

result = await hub.gateway.run_tool_loop(model, messages, tools=[get_weather_code])


# Path B — the catalog owns the definition
def get_weather(city: str) -> dict:                    # no decorator, no schema
    return {"city": city, "temp_c": 34, "sky": "hazy sun"}

rendered = await hub.prompts.render("weather-brief-catalog", "production")
result = await hub.gateway.run_prompt_with_tools(
    rendered, client_tools={"get_weather_catalog": get_weather}
)
```

Path A carries the schema in its type hints, so the repository is the source of truth. Path B
carries no schema at all — it arrives through `rendered`, and the only thing your code owns
is the function body and one map key.

Everything else you ran here is scaffolding. The preflight cell, the `__acrux_tool__` dump,
`tools.resolve` and the `walk`/`tool_spans` pair are checks; the find-or-create cells are
one-time setup, or a deploy step in the case of `tools.sync`. None of it belongs in the code
path that answers a request.

---

## Step 7 — Close the client

**Your app.** Traces are reported in the background, so close the client when you are done.
This flushes
whatever is still buffered. In a script, `async with AcruxCore() as hub:` does it for you;
in a notebook there is no block to exit, so call it yourself.

In [15]:
await hub.gateway.aclose()
print("flushed")

flushed


### Optional cleanup

Everything this notebook created lives in your team: two prompts (`weather-brief-code`,
`weather-brief-catalog`) and two tools (`get_weather_code`, `get_weather_catalog`). Leave them — the guide's
screenshots match them — or delete them from the dashboard when you are finished.

## Where to go next

- [Call a prompt's tools from the SDK](https://docs.acruxcore.com/docs/guides/call-a-prompts-tools-from-the-sdk)
  — every way to run a bound tool, including the streamed loop.
- [Manage a tool's lifecycle via the SDK](https://docs.acruxcore.com/docs/guides/manage-a-tools-lifecycle-via-the-sdk)
  — versions, aliases and `http` executors without a dashboard click.
- [Build and attach a tool](https://docs.acruxcore.com/docs/guides/build-and-attach-a-tool)
  — the code-first path end to end.